# SwarmMind — unit policy on Kaggle (bound, then BC → PPO) on the four demo maps

Thin wrapper. All logic lives in `swarmmind/training/rl/`; this notebook only finds the code,
installs it, resumes, and runs one job.

**Four maps only.** The demo plays seeds 42-45 (`assets/scenarios/demo.yaml`, `demo_seeds`)
and nothing else, so both jobs run on exactly those. A policy trained here is tuned *for those
maps* -- say "tuned on the four demo maps", never "generalises".

## Settings (right-hand panel, before running)

- **Accelerator: None.** Rollouts are the numpy simulator and the policy is a few thousand
  parameters. A GPU session gets no more CPU than a CPU session.
- **Internet: On.** The session installs the locked dependencies with `uv`.
- **Input:** the `swarmmind-rl-src` dataset (the zip from `scripts/kaggle_bundle.py`). After
  any code change, upload a new version of the dataset first.

## Run with *Save Version → Save & Run All*

Runs in the background for up to 12 hours and keeps `/kaggle/working` as the version's output.

## Jobs

- `JOB = "bound"` — ~45 min. Shipped / routing fix / routing fix + heuristic staging, on the
  four demo maps.
- `JOB = "train"` — `HOURS` of behaviour cloning from the heuristic, then PPO, on the four demo
  maps. Measured on Kaggle: one demo mission ~1,040 s with four in parallel, so one iteration
  (one episode per map) is ~17 min.

## Resuming a training run

`/kaggle/working` is **not** persistent between versions. To continue: **Add Input → Notebook
Output →** the previous version, and Save & Run All again. The resume cell finds `rl/state.pkl`
in the attached output; the trainer refuses a checkpoint trained on different seeds.

## What to send back after each training session

`rl/log.jsonl`, and at the end `rl/policy_best.npz`.


In [ ]:
# ============================== configuration ==============================
JOB = "combo"          # "bound", "train", "gate" or "combo"
HOURS = 11.0           # training budget; leaves room inside Kaggle's 12 h cap for setup
SEEDS = [42, 43, 44, 45]   # the demo maps -- must match demo.yaml `demo_seeds`
COMBO_ARMS = ["tier3", "tier3+routing"]   # + "tier3+routing+policy" to include it
LR = 1e-3             # PPO learning rate, applied on resume (session 1 ran 3e-4)
ENT_COEF = 3e-3       # entropy bonus; 1e-2 drove the drift in session 2
FROM_BEST = True      # rewind to policy_best.npz instead of the last policy


In [ ]:
import glob
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
SRC = Path("/tmp/swarmmind")            # outside /kaggle/working: keeps the saved output small
ENV = dict(os.environ, UV_PROJECT_ENVIRONMENT="/tmp/swarmmind-venv",
           UV_PYTHON_INSTALL_DIR="/tmp/uv-python", PYTHONUNBUFFERED="1")

# Kaggle unpacks an uploaded zip into a folder named after the dataset, so search rather
# than hardcode (the detector notebook's first run failed on exactly that).
roots = sorted({Path(p).parent for p in glob.glob(f"{INPUT}/**/pyproject.toml", recursive=True)
                if (Path(p).parent / "swarmmind").is_dir()})
if SRC.exists():
    shutil.rmtree(SRC)
if roots:
    shutil.copytree(roots[0], SRC)
else:
    zips = glob.glob(f"{INPUT}/**/swarmmind-rl-src.zip", recursive=True)
    assert zips, "attach the swarmmind-rl-src dataset as an input"
    with zipfile.ZipFile(zips[0]) as zf:
        zf.extractall(SRC)
print("source:", roots[0] if roots else zips[0])
print("cpus:", os.cpu_count())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "sync", "--frozen", "--python", "3.12"], cwd=SRC, env=ENV, check=True)
subprocess.run(["uv", "run", "--frozen", "python", "-c",
                "import sys, numpy; from swarmmind.sim.scenario import Scenario; "
                "print(sys.version); print('numpy', numpy.__version__); "
                "print('demo seeds', Scenario.load('demo').demo_seeds)"],
               cwd=SRC, env=ENV, check=True)


In [ ]:
def run(*args):
    """Stream a module's output into the notebook log as it happens."""
    p = subprocess.Popen(["uv", "run", "--frozen", "python", "-m", *args], cwd=SRC, env=ENV,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    code = p.wait()
    print("exit", code)
    return code


def resume(name):
    """Restore `name` from an attached previous version's output, if there is one.

    For `rl`, only a folder that actually holds `state.pkl` counts. The first version of this
    cell globbed for any folder called `rl`, which also matches `swarmmind/training/rl` inside
    the source dataset: it copied the trainer's own .py files into the output, and on a resume
    it could have picked that folder over the checkpoint and silently started from zero.
    Attach only the most recent version's output; if several are attached, the newest
    checkpoint wins.
    """
    dst = WORK / name
    if name == "rl":
        hits = [Path(h).parent for h in glob.glob(f"{INPUT}/**/rl/state.pkl", recursive=True)]
        if (dst / "state.pkl").exists() or not hits:
            print("no checkpoint attached: training starts from scratch" if not hits else
                  "checkpoint already in /kaggle/working/rl")
            return
        src = max(hits, key=lambda h: (h / "state.pkl").stat().st_mtime)
        shutil.copytree(src, dst, dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns("*.py", "__pycache__"))
    else:
        hits = glob.glob(f"{INPUT}/**/{name}", recursive=True)
        if dst.exists() or not hits:
            return
        src = Path(max(hits, key=os.path.getmtime))
        shutil.copy(src, dst)
    print("resumed", name, "from", src)


In [ ]:
seeds = [str(s) for s in SEEDS]
if JOB == "bound":
    resume("bound.jsonl")
    run("swarmmind.training.rl.bound", "--out", str(WORK / "bound.jsonl"), "--seeds", *seeds)
elif JOB == "train":
    resume("rl")
    run("swarmmind.training.rl.run", "--hours", str(HOURS), "--out", str(WORK / "rl"),
        "--seeds", *seeds, "--lr", str(LR), "--ent-coef", str(ENT_COEF),
        *(["--from-best"] if FROM_BEST else []))
elif JOB == "gate":
    # The ship / don't-ship report, from real missions on the demo maps. Attach the last
    # training version's output so `rl/policy_best.npz` is there to gate.
    resume("rl")
    run("swarmmind.training.gate", "--scenario", "demo", "--workers", "4",
        "--unit-policy", str(WORK / "rl" / "policy_best.npz"),
        "--report", str(WORK / "SHIPPING.md"))
elif JOB == "combo":
    # What the demo would actually run: Tier 3 live, with and without the routing fix.
    # ~40 min for two arms on four maps. Attach the training output only if an arm needs
    # the trained policy.
    resume("rl")
    run("swarmmind.training.rl.bound", "--out", str(WORK / "combo.jsonl"),
        "--seeds", *seeds, "--arms", *COMBO_ARMS,
        "--unit-policy", str(WORK / "rl" / "policy_best.npz"))
else:
    raise ValueError(JOB)


In [ ]:
for p in sorted(WORK.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / 1e6:8.2f} MB  {p.relative_to(WORK)}")
